In [13]:
import sys
print(sys.executable)
import pandas as pd

/home/maria/anaconda3/envs/rbpbench/bin/python


In [2]:
!ls /home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/miRNA

intellectual_disability_negative_PSI
intellectual_disability_PCB153-vs-CTR_ASE_specific_coordinates.bed
intellectual_disability_PCB153-vs-CTR_ASE_specific_coordinates.fa
intellectual_disability_PCB153-vs-CTR_specific_coordinates_DASE.txt
intellectual_disability_PCB153-vs-CTR_specific_coordinates_new_portions.csv
intellectual_disability_positive_PSI
negative_PSI
PCB153-vs-CTR_ASE_specific_coordinates.bed
PCB153-vs-CTR_ASE_specific_coordinates.fa
PCB153-vs-CTR_specific_coordinates_DASE.txt
PCB153-vs-CTR_specific_coordinates_new_portions.csv
positive_PSI


In [3]:
!ls /home/maria/Desktop/progetto_inquinanti/RMATS_RESULTS/GTF_GRCh38

GRCh38.primary_assembly.genome.chr_only_clean_2.fa
GRCh38.primary_assembly.genome.chr_only_clean_2.fa.fai
GRCh38.primary_assembly.genome.chr_only.fa
GRCh38.primary_assembly.genome.chr_only.fa.fai
GRCh38.primary_assembly.genome.fa
GRCh38.primary_assembly.genome.fa.fai
Homo_sapiens.GRCh38.112.gtf


In [4]:
#!awk '/^>/ {keep=($1~/^>chr/)} keep' /home/maria/Desktop/progetto_inquinanti/RMATS_RESULTS/GTF_GRCh38/GRCh38.primary_assembly.genome.fa > /home/maria/Desktop/progetto_inquinanti/RMATS_RESULTS/GTF_GRCh38/GRCh38.primary_assembly.genome.chr_only.fa


In [5]:
#!grep '>' /home/maria/Desktop/progetto_inquinanti/RMATS_RESULTS/GTF_GRCh38/GRCh38.primary_assembly.genome.chr_only.fa

In [6]:
#!ls /home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/miRNA/*bed

In [7]:
!ls /home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/miRNA/negative_PSI/*bed

/home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/miRNA/negative_PSI/intellectual_disability_negative_PSI_PCB153-vs-CTR_ASE_specific_coordinates.bed
/home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/miRNA/negative_PSI/negative_PSI_PCB153-vs-CTR_ASE_specific_coordinates.bed


In [8]:
!ls /home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/miRNA/positive_PSI/*bed

/home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/miRNA/positive_PSI/intellectual_disability_positive_PSI_PCB153-vs-CTR_ASE_specific_coordinates.bed
/home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/miRNA/positive_PSI/positive_PSI_PCB153-vs-CTR_ASE_specific_coordinates.bed


In [9]:
#!grep '>' /home/maria/Desktop/progetto_inquinanti/RMATS_RESULTS/GTF_GRCh38/GRCh38.primary_assembly.genome.chr_only_clean_2.fa

In [10]:
!pwd

/home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/RBP-DEGS/RBPbench


In [17]:
# ==============================
# Specify a custom list of RBPs
# ==============================

rbp_deg=pd.read_csv('/home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/RBP-DEGS/PCB153-vs-CTR-RBP-DEGS.txt')
specific_rbps =rbp_deg.Gene.to_list()


In [18]:
specific_rbps

['ADAR',
 'CELF2',
 'CNOT4',
 'DDX3X',
 'EIF4G2',
 'ERI1',
 'G3BP2',
 'HNRNPC',
 'HNRNPH2',
 'HNRNPK',
 'HNRNPLL',
 'IGF2BP3',
 'ILF3',
 'NONO',
 'NUMA1',
 'PABPC5',
 'PRPF8',
 'PTBP2',
 'RBFOX1',
 'RBM14',
 'RBM22',
 'SART3',
 'SLTM',
 'SND1',
 'SNRPB2',
 'SRSF3',
 'SRSF9',
 'SUB1',
 'TNRC6A',
 'UCHL5',
 'YBX1',
 'YTHDC1']

In [20]:
import os
import subprocess
from glob import glob

# ==============================
# Full path to rbpbench executable
# ==============================
RBP_PATH = "/home/maria/anaconda3/envs/rbpbench/bin/rbpbench"

# ==============================
# Main output directory
# ==============================
main_out = "/home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/RBP-DEGS/RBPbench"
specific_out_base = os.path.join(main_out, "RBP_DEG_specific")
os.makedirs(specific_out_base, exist_ok=True)

# ==============================
# Input BED directories
# ==============================
miRNA_dir = "/home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/miRNA"
positive_dir = os.path.join(miRNA_dir, "intellectual_disability_positive_PSI")
negative_dir = os.path.join(miRNA_dir, "intellectual_disability_negative_PSI")

# Genome and GTF paths
genome_search = "/home/maria/Desktop/progetto_inquinanti/RMATS_RESULTS/GTF_GRCh38/GRCh38.primary_assembly.genome.chr_only.fa"
genome_enmo = "/home/maria/Desktop/progetto_inquinanti/RMATS_RESULTS/GTF_GRCh38/GRCh38.primary_assembly.genome.chr_only_clean_2.fa"
gtf = "/home/maria/Desktop/progetto_inquinanti/RMATS_RESULTS/GTF_GRCh38/Homo_sapiens.GRCh38.112.gtf"

# ==============================
# Helper function to run rbpbench
# ==============================
def run_rbpbench(mode, bed_file, out_dir, rbps):
    # Build command
    cmd = [
        RBP_PATH,
        mode,
        "--in", bed_file,
        "--genome", genome_search if mode=="search" else genome_enmo,
        "--gtf", gtf,
        "--out", out_dir,
        "--rbps"
    ] + rbps  # <- pass RBPs as separate arguments
    cmd += ["--fisher-mode", "2", "--cooc-pval-mode", "1"]

    if mode == "search":
        cmd += ["--enable-upset-plot", "--plot-motifs", "--ext", "250", "--fimo-pval", "0.0005", "--wrs-mode", "1"]
    elif mode == "enmo":
        cmd += ["--fimo-pval", "0.0005", "--ext", "250", "--enmo-pval-thr", "0.05"]

    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

# ==============================
# Find all BED files
# ==============================
positive_beds = glob(os.path.join(positive_dir, "*.bed"))
negative_beds = glob(os.path.join(negative_dir, "*.bed"))

# ==============================
# Function to process a list of BED files with specific RBPs
# ==============================
def process_beds_specific(bed_list, out_base, rbps):
    for bed_file in bed_list:
        bed_name = os.path.basename(bed_file)
        base_name = bed_name.replace(".bed", "")

        out_search_spec = os.path.join(out_base, f"{base_name}_search")
        out_enmo_spec = os.path.join(out_base, f"{base_name}_enmo")
        os.makedirs(out_search_spec, exist_ok=True)
        os.makedirs(out_enmo_spec, exist_ok=True)

        # Run only for specific RBPs
        run_rbpbench("search", bed_file, out_search_spec, rbps)
        run_rbpbench("enmo", bed_file, out_enmo_spec, rbps)

# ==============================
# Example: specify RBPs
# ==============================


# ==============================
# Process BED files
# ==============================
process_beds_specific(positive_beds, specific_out_base, specific_rbps)
process_beds_specific(negative_beds, specific_out_base, specific_rbps)


Running: /home/maria/anaconda3/envs/rbpbench/bin/rbpbench search --in /home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/miRNA/intellectual_disability_positive_PSI/intellectual_disability_positive_PSI_PCB153-vs-CTR_ASE_specific_coordinates.bed --genome /home/maria/Desktop/progetto_inquinanti/RMATS_RESULTS/GTF_GRCh38/GRCh38.primary_assembly.genome.chr_only.fa --gtf /home/maria/Desktop/progetto_inquinanti/RMATS_RESULTS/GTF_GRCh38/Homo_sapiens.GRCh38.112.gtf --out /home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/RBP-DEGS/RBPbench/RBP_DEG_specific/intellectual_disability_positive_PSI_PCB153-vs-CTR_ASE_specific_coordinates_search --rbps ADAR CELF2 CNOT4 DDX3X EIF4G2 ERI1 G3BP2 HNRNPC HNRNPH2 HNRNPK HNRNPLL IGF2BP3 ILF3 NONO NUMA1 PABPC5 PRPF8 PTBP2 RBFOX1 RBM14 RBM22 SART3 SLTM SND1 SNRPB2 SRSF3 SRSF9 SUB1 TNRC6A UCHL5 YBX1 YTHDC1 --fisher-mode 2 --cooc-pval-mode 1 --enable-upset-plot --plot-motifs --ext 250 --fimo-pval 0.0005 --wrs-mode 1



rbpbench search --in /home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/miRNA/PCB153-vs-CTR_ASE_specific_coordinates.bed --genome /home/maria/Desktop/progetto_inquinanti/RMATS_RESULTS/GTF_GRCh38/GRCh38.primary_assembly.genome.chr_only.fa --gtf /home/maria/Desktop/progetto_inquinanti/RMATS_RESULTS/GTF_GRCh38/Homo_sapiens.GRCh38.112.gtf --out RBPENC_output --rbps ALL --enable-upset-plot --plot-motifs --fimo-pval 0.0001 --wrs-mode 1 --fisher-mode 2 --cooc-pval-mode 1


rbpbench enmo --in /home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/miRNA/PCB153-vs-CTR_ASE_specific_coordinates.bed --genome /home/maria/Desktop/progetto_inquinanti/RMATS_RESULTS/GTF_GRCh38/GRCh38.primary_assembly.genome.chr_only_clean_2.fa --gtf /home/maria/Desktop/progetto_inquinanti/RMATS_RESULTS/GTF_GRCh38/Homo_sapiens.GRCh38.112.gtf --out RBPENC_output_enmo --rbps ALL --fimo-pval 0.0005 --fisher-mode 2 --cooc-pval-mode 1 --ext 250 --enmo-pval-thr 0.05